# RL Phase 2 from step-150: Gold-aware Hard Negatives + Endpoint Reward

This notebook continues from the existing RL `step-150` adapter with a new training branch.

### Phase-2 changes

- Source weights: `checkpoint-1200-rl-step-150`
- Fresh optimizer: `PagedAdamW8bit`, learning rate `3e-7`
- Rollout set per train example:
  - gold order × 1
  - current-policy top-4 non-gold orders × 4
  - policy-weighted random remaining non-gold order × 1
- Reward:
  - exact `0.50`
  - first `0.10`
  - last `0.10`
  - both endpoints `0.20`
  - adjacency `0.05`
  - relative pair accuracy `0.05`
- Train for 100 additional phase steps
- Save and run a fixed matched-20 quick evaluation every 25 phase steps
- Quick evaluation records full-order and partial confidence metrics

The new sampler is **gold-aware hard-negative policy optimization**, rather than purely stochastic on-policy rollout sampling.

In [ ]:
# 1) Install dependencies, then restart runtime once.
# After restart, run this cell again and continue.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import subprocess
import sys
from pathlib import Path

MARKER = Path("/content/.snu_pilot_c_deps_installed")

if Path("/content").exists() and not MARKER.exists():
    packages = [
        "transformers>=4.49.0,<4.54.0",
        "accelerate>=0.34.0",
        "bitsandbytes>=0.46.1",
        "peft",
        "qwen-vl-utils",
        "huggingface_hub",
        "hf_xet",
        "modelscope",
        "jedi",
        "pandas==2.2.2",
        "safetensors>=0.4.5",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])
    MARKER.write_text("ok")
    print("Dependencies installed. Restarting runtime. Run this cell again after restart.")
    os.kill(os.getpid(), 9)
elif Path("/content").exists():
    print("Dependencies already installed. Continue.")
else:
    print("Local environment detected. Skipping Colab dependency install.")


In [ ]:
# 2) Setup: Drive, data, model cache, run paths
from google.colab import drive
from pathlib import Path
import os
import shutil

drive_root = Path("/content/drive")
if drive_root.exists() and not os.path.ismount(str(drive_root)) and any(drive_root.iterdir()):
    print("Removing local pre-mount /content/drive contents:", sorted(str(p) for p in drive_root.iterdir())[:20])
    shutil.rmtree(drive_root)
drive_root.mkdir(parents=True, exist_ok=True)
drive.mount("/content/drive")

import ast
import copy
import gc
import glob
import itertools
import json
import math
import random
import re
import subprocess
import zipfile
from datetime import datetime

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig, Trainer, TrainingArguments, TrainerCallback, set_seed
try:
    from transformers import Qwen2VLForConditionalGeneration
except ImportError:
    Qwen2VLForConditionalGeneration = AutoModelForVision2Seq
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers.utils import logging as transformers_logging

transformers_logging.set_verbosity_error()

SNU_ROOT = Path("/content/drive/MyDrive/SNU_AI_Challenge")
assert SNU_ROOT.exists(), SNU_ROOT

ZIP_PATH = SNU_ROOT / "snuaichallenge.zip"
DATA_DIR = Path("/content/snuaichallenge_data")
TRAIN_CSV = DATA_DIR / "train.csv"
TEST_CSV = DATA_DIR / "test.csv"
TRAIN_IMAGE_DIR = DATA_DIR / "train"
TEST_IMAGE_DIR = DATA_DIR / "test"

if not TRAIN_CSV.exists() or not TRAIN_IMAGE_DIR.is_dir():
    print("Extracting:", ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as zip_file:
        zip_file.extractall("/content/")

assert TRAIN_CSV.exists(), TRAIN_CSV
assert TEST_CSV.exists(), TEST_CSV
assert TRAIN_IMAGE_DIR.is_dir(), TRAIN_IMAGE_DIR
assert TEST_IMAGE_DIR.is_dir(), TEST_IMAGE_DIR

MODEL_REPO_ID = "Qwen/Qwen2-VL-7B-Instruct"
USE_MODELSCOPE_BASE_MODEL = False
DRIVE_MODEL_DIR = SNU_ROOT / "model_cache/Qwen2-VL-7B-Instruct"
LOCAL_MODEL_DIR = Path("/content/Qwen2-VL-7B-Instruct")


def print_runtime_storage():
    print("Storage check for /content:")
    try:
        subprocess.run(["df", "-h", "/content"], check=False)
    except Exception as exc:
        total, used, free = shutil.disk_usage("/content")
        print(f"/content free: {free / (1024 ** 3):.1f} GB / total: {total / (1024 ** 3):.1f} GB ({exc})")

    try:
        meminfo = {}
        with open("/proc/meminfo", "r", encoding="utf-8") as handle:
            for line in handle:
                key, value = line.split(":", 1)
                meminfo[key] = int(value.strip().split()[0]) / (1024 ** 2)
        print(f"RAM available: {meminfo.get('MemAvailable', 0):.1f} GB / total: {meminfo.get('MemTotal', 0):.1f} GB")
    except Exception as exc:
        print("RAM check skipped:", exc)

    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        print(f"GPU memory free: {free / (1024 ** 3):.1f} GB / total: {total / (1024 ** 3):.1f} GB")


def model_cache_is_complete(model_dir):
    model_dir = Path(model_dir)
    if not (model_dir / "config.json").exists():
        return False
    has_weight = (model_dir / "model.safetensors.index.json").exists() or bool(list(model_dir.glob("*.safetensors")))
    has_processor = any((model_dir / name).exists() for name in ["preprocessor_config.json", "processor_config.json", "tokenizer.json", "tokenizer_config.json"])
    return bool(has_weight and has_processor)


def copy_drive_cache_to_local():
    if not model_cache_is_complete(DRIVE_MODEL_DIR):
        raise FileNotFoundError(f"Drive model cache is incomplete or missing: {DRIVE_MODEL_DIR}")

    if model_cache_is_complete(LOCAL_MODEL_DIR):
        print("Using existing local model:", LOCAL_MODEL_DIR)
        return

    print("Copying base model from Drive to Colab local disk...")
    print("  from:", DRIVE_MODEL_DIR)
    print("  to  :", LOCAL_MODEL_DIR)
    if LOCAL_MODEL_DIR.exists():
        shutil.rmtree(LOCAL_MODEL_DIR)
    shutil.copytree(DRIVE_MODEL_DIR, LOCAL_MODEL_DIR)
    assert model_cache_is_complete(LOCAL_MODEL_DIR), LOCAL_MODEL_DIR


def download_base_model_to_local_and_cache():
    print("Base model cache not found. Downloading with Hugging Face snapshot_download to local disk first:")
    print("repo:", MODEL_REPO_ID)
    if LOCAL_MODEL_DIR.exists() and not model_cache_is_complete(LOCAL_MODEL_DIR):
        shutil.rmtree(LOCAL_MODEL_DIR)
    from huggingface_hub import snapshot_download
    hf_token = os.environ.get("HF_TOKEN")
    model_dir = snapshot_download(
        repo_id=MODEL_REPO_ID,
        local_dir=str(LOCAL_MODEL_DIR),
        token=hf_token,
        max_workers=8,
    )
    print("Downloaded:", model_dir)
    assert model_cache_is_complete(LOCAL_MODEL_DIR), LOCAL_MODEL_DIR

    DRIVE_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(DRIVE_MODEL_DIR) + ".tmp")
    if tmp.exists():
        shutil.rmtree(tmp)
    shutil.copytree(LOCAL_MODEL_DIR, tmp)
    if DRIVE_MODEL_DIR.exists():
        shutil.rmtree(DRIVE_MODEL_DIR)
    os.replace(tmp, DRIVE_MODEL_DIR)
    print("Saved to Drive:", DRIVE_MODEL_DIR)


def ensure_base_model_path():
    print_runtime_storage()

    if model_cache_is_complete(LOCAL_MODEL_DIR):
        print("Using local model:", LOCAL_MODEL_DIR)
    elif model_cache_is_complete(DRIVE_MODEL_DIR):
        copy_drive_cache_to_local()
    elif not USE_MODELSCOPE_BASE_MODEL:
        download_base_model_to_local_and_cache()
    else:
        print("Base model cache not found. Downloading via ModelScope:", MODEL_REPO_ID)
        from modelscope import snapshot_download as modelscope_snapshot_download
        model_dir = modelscope_snapshot_download(MODEL_REPO_ID, cache_dir="/content/modelscope_cache")
        if LOCAL_MODEL_DIR.exists():
            shutil.rmtree(LOCAL_MODEL_DIR)
        shutil.copytree(model_dir, LOCAL_MODEL_DIR)
        assert model_cache_is_complete(LOCAL_MODEL_DIR), LOCAL_MODEL_DIR
        DRIVE_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
        tmp = Path(str(DRIVE_MODEL_DIR) + ".tmp")
        if tmp.exists():
            shutil.rmtree(tmp)
        shutil.copytree(LOCAL_MODEL_DIR, tmp)
        if DRIVE_MODEL_DIR.exists():
            shutil.rmtree(DRIVE_MODEL_DIR)
        os.replace(tmp, DRIVE_MODEL_DIR)
        print("Saved to Drive:", DRIVE_MODEL_DIR)

    print_runtime_storage()
    print("Using local model:", LOCAL_MODEL_DIR)
    return str(LOCAL_MODEL_DIR)



MODEL_ID = ensure_base_model_path()
MODEL_LOCAL_FILES_ONLY = True

OUTPUT_ROOT = SNU_ROOT / "qwen2vl_7b_multitask_bipair_conditional_v1"

# Original SFT checkpoint metadata
BASE_RUN_ID = "20260721_011312"
BASE_CHECKPOINT_NAME = "checkpoint-1200"
BASE_RUN_ROOT = OUTPUT_ROOT / "runs" / BASE_RUN_ID
BASE_OUTPUT_DIR = BASE_RUN_ROOT / "multitask_bipair_conditional"
BASE_CHECKPOINT_DIR = BASE_OUTPUT_DIR / BASE_CHECKPOINT_NAME
assert BASE_CHECKPOINT_DIR.is_dir(), BASE_CHECKPOINT_DIR

# Phase-1 RL source: continue weights from step 150.
SOURCE_RL_RUN_ID = "20260721_164630_rl_from_checkpoint1200"
SOURCE_RL_OUTPUT_DIR = (
    OUTPUT_ROOT
    / "runs"
    / SOURCE_RL_RUN_ID
    / "onpolicy_rl_from_checkpoint1200"
)
SOURCE_RL_CHECKPOINT_DIR = (
    SOURCE_RL_OUTPUT_DIR
    / "checkpoints"
    / "checkpoint-1200-rl-step-150"
)
assert SOURCE_RL_CHECKPOINT_DIR.is_dir(), SOURCE_RL_CHECKPOINT_DIR

# Stable branch ID so a disconnected runtime can auto-resume this phase.
PHASE2_RUN_ID = "20260722_rl_phase2_from_step150_goldhard_reward_v1"
RUN_ID = PHASE2_RUN_ID
RUN_ROOT = OUTPUT_ROOT / "runs" / RUN_ID
OUTPUT_DIR = RUN_ROOT / "goldhard_endpoint_reward_phase2"
EVAL_DIR = OUTPUT_DIR / "eval"
QUICK_EVAL_DIR = EVAL_DIR / "quick20_partial_confidence"
CHECKPOINT_ROOT = OUTPUT_DIR / "checkpoints"
for path in [RUN_ROOT, OUTPUT_DIR, EVAL_DIR, QUICK_EVAL_DIR, CHECKPOINT_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

SEED = 42
VALID_RATIO = 0.10
TRAIN_ROWS = None
VALID_ROWS = None

# Fixed quick set: first 20 examples of the matched seed-43 validation 100.
MATCHED_EVAL_SEED = SEED + 1
MATCHED_POOL_ROWS = 100
QUICK_EVAL_ROWS = 20
RUN_INITIAL_QUICK_EVAL = True
RUN_QUICK_EVAL_DURING_TRAIN = True
FORCE_QUICK_REEVAL = False

TASK_RATIOS = {
    "order": 0.30,
    "pairwise": 0.25,
    "first": 0.10,
    "last": 0.10,
    "fixed_first": 0.10,
    "fixed_last": 0.10,
    "fixed_endpoints": 0.05,
}
TASK_LOSS_WEIGHTS = {task: 1.0 for task in TASK_RATIOS}

# Adapter architecture is inherited from the source checkpoint.
LORA_R = 64
LORA_ALPHA = 128
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# Phase-2 training configuration
SOURCE_ABSOLUTE_RL_STEP = 150
LEARNING_RATE = 3e-7
RL_STEPS = 100                 # additional phase steps
ROLLOUTS_PER_SAMPLE = 6
HARD_NEGATIVES_PER_SAMPLE = 4
RANDOM_NEGATIVES_PER_SAMPLE = 1
TEMPERATURE = 0.8
SFT_REPLAY_WEIGHT = 0.20
MAX_GRAD_NORM = 1.0
SAVE_STEPS = 25
EVAL_STEPS = 25
LOGGING_STEPS = 5
AUTO_RESUME_PHASE2 = True

# Train-row hard cache is still disabled. This notebook applies hard-negative
# mining within each selected train example, not across the train dataset.
HARD_SAMPLE_RATIO = 0.00
RANDOM_SAMPLE_RATIO = 1.00
TRAIN_HARD_CACHE_PATH = None

# T4-safe micro-batches
SFT_REPLAY_BATCH_SIZE = 1
RL_CANDIDATE_BATCH_SIZE = 1
RL_GRAD_CANDIDATE_BATCH_SIZE = 1
QUICK_EVAL_CANDIDATE_BATCH_SIZE = 1

# Reward weights
REWARD_WEIGHTS = {
    "exact": 0.50,
    "first": 0.10,
    "last": 0.10,
    "both_endpoints": 0.20,
    "adjacency": 0.05,
    "pair": 0.05,
}
assert abs(sum(REWARD_WEIGHTS.values()) - 1.0) < 1e-9

MIN_PIXELS = 128 * 28 * 28
MAX_PIXELS = 256 * 28 * 28
PAIR_INDICES = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
PERMUTATIONS = list(itertools.permutations([1, 2, 3, 4]))

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("SFT base checkpoint:", BASE_CHECKPOINT_DIR)
print("Phase-1 source adapter:", SOURCE_RL_CHECKPOINT_DIR)
print("Phase-2 run root:", RUN_ROOT)
print("Phase-2 output:", OUTPUT_DIR)
print("Model:", MODEL_ID)


In [ ]:
# 3) Data split and Pilot C multitask record generation
def parse_answer(answer):
    result = answer if isinstance(answer, list) else ast.literal_eval(str(answer))
    result = [int(value) for value in result]
    if len(result) != 4 or sorted(result) != [1, 2, 3, 4]:
        raise ValueError(f"Invalid Answer: {answer}")
    return result


def order_to_sequence(answer):
    return [input_index + 1 for input_index, _ in sorted(enumerate(answer), key=lambda item: item[1])]


def compact_order(order):
    return " ".join(str(int(value)) for value in order)


def parse_compact_order(text, expected_len=4):
    values = [int(x) for x in re.findall(r"[1-4]", str(text))]
    if len(values) != expected_len or len(set(values)) != expected_len:
        return None
    return values


def row_image_paths(row, image_root=TRAIN_IMAGE_DIR):
    sample_id = str(row["Id"])
    return [str(Path(image_root) / sample_id / str(row[f"Input_{i}"])) for i in range(1, 5)]


def load_rgb(path):
    with Image.open(path) as image:
        return image.convert("RGB").copy()


def base_record(row_index, row):
    answer = [int(value) for value in row["Answer_list"]]
    order = order_to_sequence(answer)
    return {
        "row_index": int(row_index),
        "sample_id": str(row["Id"]),
        "sentence": "" if pd.isna(row["Sentence"]) else str(row["Sentence"]),
        "answer": answer,
        "order": order,
        "image_paths": row_image_paths(row, TRAIN_IMAGE_DIR),
    }


def pair_target_for_order(order, a, b):
    ranks = {frame: idx for idx, frame in enumerate(order)}
    return "A" if ranks[int(a)] < ranks[int(b)] else "B"


def build_pair_groups(base):
    groups = []
    order = base["order"]
    for i, j in PAIR_INDICES:
        a, b = i + 1, j + 1
        group = []
        for left, right in [(a, b), (b, a)]:
            item = copy.deepcopy(base)
            item.update({
                "task_type": "pairwise",
                "pair": [left, right],
                "image_paths": [base["image_paths"][left - 1], base["image_paths"][right - 1]],
                "target": pair_target_for_order(order, left, right),
            })
            group.append(item)
        groups.append(group)
    return groups


def build_record_pools(dataframe):
    pools = {task: [] for task in TASK_RATIOS}
    pools["pairwise_groups"] = []
    for row_index, row in dataframe.iterrows():
        base = base_record(row_index, row)
        order = base["order"]

        item = copy.deepcopy(base)
        item.update({"task_type": "order", "target": compact_order(order)})
        pools["order"].append(item)

        item = copy.deepcopy(base)
        item.update({"task_type": "first", "target": str(order[0])})
        pools["first"].append(item)

        item = copy.deepcopy(base)
        item.update({"task_type": "last", "target": str(order[-1])})
        pools["last"].append(item)

        pair_groups = build_pair_groups(base)
        pools["pairwise_groups"].extend(pair_groups)
        for group in pair_groups:
            pools["pairwise"].extend(group)

        first = order[0]
        remaining = [x for x in order if x != first]
        item = copy.deepcopy(base)
        item.update({"task_type": "fixed_first", "fixed_first": first, "target": compact_order(remaining)})
        pools["fixed_first"].append(item)

        last = order[-1]
        remaining = [x for x in order if x != last]
        item = copy.deepcopy(base)
        item.update({"task_type": "fixed_last", "fixed_last": last, "target": compact_order(remaining)})
        pools["fixed_last"].append(item)

        middle = [x for x in order if x not in {order[0], order[-1]}]
        item = copy.deepcopy(base)
        item.update({"task_type": "fixed_endpoints", "fixed_first": order[0], "fixed_last": order[-1], "target": compact_order(middle)})
        pools["fixed_endpoints"].append(item)
    return pools


def sample_records(records, count, rng):
    records = list(records)
    if count <= len(records):
        indices = rng.choice(len(records), size=count, replace=False)
    else:
        base_indices = np.arange(len(records))
        extra_indices = rng.choice(len(records), size=count - len(records), replace=True)
        indices = np.concatenate([base_indices, extra_indices])
        rng.shuffle(indices)
    return [records[int(index)] for index in indices]


def sample_pair_groups(groups, target_record_count, rng):
    groups = list(groups)
    group_count = max(1, int(math.ceil(target_record_count / 2)))
    selected_groups = sample_records(groups, group_count, rng)
    records = [record for group in selected_groups for record in group]
    return records[:target_record_count] if len(records) > target_record_count else records


def build_balanced_records(dataframe):
    pools = build_record_pools(dataframe)
    base_total = int(math.ceil(len(pools["order"]) / TASK_RATIOS["order"]))
    rng = np.random.default_rng(SEED)
    merged = []
    distribution = {}
    for task, ratio in TASK_RATIOS.items():
        count = max(1, int(round(base_total * ratio)))
        if task == "pairwise":
            records = sample_pair_groups(pools["pairwise_groups"], count, rng)
            distribution[task] = {
                "pair_group_pool": len(pools["pairwise_groups"]),
                "record_pool": len(pools["pairwise"]),
                "sampled": len(records),
                "sampled_groups": int(math.ceil(len(records) / 2)),
            }
        else:
            records = sample_records(pools[task], count, rng)
            distribution[task] = {"pool": len(pools[task]), "sampled": len(records)}
        merged.extend(records)
        print(task, distribution[task])
    rng.shuffle(merged)
    return merged, pools, distribution

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
train_df["Id"] = train_df["Id"].astype(str)
test_df["Id"] = test_df["Id"].astype(str)
train_df["Answer_list"] = train_df["Answer"].apply(parse_answer)

split_root = SNU_ROOT / "id_splits" / "qwen2vl_lgt_order_refine_20260714_003635"
if (split_root / "train_ids.json").exists() and (split_root / "validation_ids.json").exists():
    train_ids = set(str(x) for x in json.load(open(split_root / "train_ids.json", "r", encoding="utf-8")))
    valid_ids = set(str(x) for x in json.load(open(split_root / "validation_ids.json", "r", encoding="utf-8")))
else:
    unique_ids = train_df["Id"].unique().copy()
    rng = np.random.default_rng(SEED)
    rng.shuffle(unique_ids)
    valid_size = max(1, int(len(unique_ids) * VALID_RATIO))
    valid_ids = set(unique_ids[:valid_size])
    train_ids = set(unique_ids[valid_size:])

training_df = train_df[train_df["Id"].isin(train_ids)].reset_index(drop=True)
validation_df = train_df[train_df["Id"].isin(valid_ids)].reset_index(drop=True)
if TRAIN_ROWS is not None:
    training_df = training_df.sample(n=min(TRAIN_ROWS, len(training_df)), random_state=SEED).reset_index(drop=True)
if VALID_ROWS is not None:
    validation_df = validation_df.sample(n=min(VALID_ROWS, len(validation_df)), random_state=SEED).reset_index(drop=True)

train_records, train_pools, task_distribution = build_balanced_records(training_df)
valid_pools = build_record_pools(validation_df)
matched_pool_df = validation_df.sample(
    n=min(MATCHED_POOL_ROWS, len(validation_df)),
    random_state=MATCHED_EVAL_SEED,
)
quick_eval_df = matched_pool_df.head(
    min(QUICK_EVAL_ROWS, len(matched_pool_df))
).copy()
quick_eval_ids = quick_eval_df["Id"].astype(str).tolist()
quick_eval_sample_hash = __import__("hashlib").sha1(
    "\n".join(quick_eval_ids).encode("utf-8")
).hexdigest()
pd.DataFrame({
    "quick_position": range(1, len(quick_eval_ids) + 1),
    "sample_id": quick_eval_ids,
}).to_csv(QUICK_EVAL_DIR / "quick20_sample_ids.csv", index=False)

run_config = {
    "experiment": "rl_phase2_from_step150_goldhard_endpoint_reward",
    "base_run_id": BASE_RUN_ID,
    "base_checkpoint_name": BASE_CHECKPOINT_NAME,
    "base_checkpoint_dir": str(BASE_CHECKPOINT_DIR),
    "source_rl_run_id": SOURCE_RL_RUN_ID,
    "source_rl_checkpoint_dir": str(SOURCE_RL_CHECKPOINT_DIR),
    "phase2_run_id": RUN_ID,
    "output_dir": str(OUTPUT_DIR),
    "task_ratios": TASK_RATIOS,
    "task_loss_weights": TASK_LOSS_WEIGHTS,
    "task_distribution": task_distribution,
    "learning_rate": LEARNING_RATE,
    "phase2_steps": RL_STEPS,
    "source_absolute_rl_step": SOURCE_ABSOLUTE_RL_STEP,
    "rollouts_per_sample": ROLLOUTS_PER_SAMPLE,
    "hard_negatives_per_sample": HARD_NEGATIVES_PER_SAMPLE,
    "random_negatives_per_sample": RANDOM_NEGATIVES_PER_SAMPLE,
    "rollout_policy": "gold + top4 non-gold + policy-weighted random non-gold",
    "reward_weights": REWARD_WEIGHTS,
    "temperature": TEMPERATURE,
    "sft_replay_weight": SFT_REPLAY_WEIGHT,
    "save_steps": SAVE_STEPS,
    "eval_steps": EVAL_STEPS,
    "quick_eval_rows": len(quick_eval_df),
    "quick_eval_seed": MATCHED_EVAL_SEED,
    "quick_eval_sample_hash": quick_eval_sample_hash,
    "seed": SEED,
    "train_rows": len(training_df),
    "validation_rows": len(validation_df),
}
with open(RUN_ROOT / "run_config.json", "w", encoding="utf-8") as f:
    json.dump(run_config, f, ensure_ascii=False, indent=2)
with open(OUTPUT_DIR / "task_distribution_rl.json", "w", encoding="utf-8") as f:
    json.dump(task_distribution, f, ensure_ascii=False, indent=2)

print("quick20 seed/hash:", MATCHED_EVAL_SEED, quick_eval_sample_hash)
print("quick20 first IDs:", quick_eval_ids[:10])

print("train/valid/test:", len(training_df), len(validation_df), len(test_df))
print("train records:", len(train_records))


In [ ]:
# 4) Prompt builders, dataset, collator
def task_instruction(example):
    sentence = example["sentence"]
    task_type = example["task_type"]
    if task_type == "pairwise":
        return (
            f"Caption:\n{sentence}\n\n"
            "The two candidate images are labeled A and B in the presented order.\n"
            "Which image occurs earlier in the story timeline?\n"
            "Answer only A or B."
        )
    if task_type == "first":
        return f"Caption:\n{sentence}\n\nWhich image is the first scene in the story? Answer only one frame number from 1 to 4."
    if task_type == "last":
        return f"Caption:\n{sentence}\n\nWhich image is the last scene in the story? Answer only one frame number from 1 to 4."
    if task_type == "order":
        return (
            f"Caption:\n{sentence}\n\n"
            "Order all four images from earliest to latest in the story.\n"
            "Answer only four frame numbers separated by spaces, for example: 1 2 3 4."
        )
    if task_type == "fixed_first":
        return (
            f"Caption:\n{sentence}\n\n"
            f"Frame {example['fixed_first']} is fixed as the first scene.\n"
            "Order the remaining frames from earliest to latest.\n"
            "Answer only the remaining frame numbers separated by spaces."
        )
    if task_type == "fixed_last":
        return (
            f"Caption:\n{sentence}\n\n"
            f"Frame {example['fixed_last']} is fixed as the last scene.\n"
            "Order the remaining frames from earliest to latest.\n"
            "Answer only the remaining frame numbers separated by spaces."
        )
    if task_type == "fixed_endpoints":
        return (
            f"Caption:\n{sentence}\n\n"
            f"Frame {example['fixed_first']} is fixed as the first scene.\n"
            f"Frame {example['fixed_last']} is fixed as the last scene.\n"
            "Order the remaining middle frames from earliest to latest.\n"
            "Answer only the remaining frame numbers separated by spaces."
        )
    raise ValueError(task_type)


def make_messages(example, include_answer=False):
    content = []
    for idx, _ in enumerate(example["image_paths"], start=1):
        label = "A" if example["task_type"] == "pairwise" and idx == 1 else "B" if example["task_type"] == "pairwise" and idx == 2 else str(idx)
        content.append({"type": "text", "text": f"\nImage {label}:"})
        content.append({"type": "image"})
    content.append({"type": "text", "text": "\n\n" + task_instruction(example)})
    messages = [{"role": "user", "content": content}]
    if include_answer:
        messages.append({"role": "assistant", "content": str(example["target"])})
    return messages


class PilotCDataset(Dataset):
    def __init__(self, records):
        self.records = list(records)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        return self.records[index]


class PilotCCollator:
    def __init__(self, processor):
        self.processor = processor
        self.tokenizer = processor.tokenizer
        self.assistant_prefix_ids = self.tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)

    def _mask_prompt(self, input_ids):
        ids = input_ids.tolist()
        labels = input_ids.clone()
        start = None
        prefix = self.assistant_prefix_ids
        for i in range(0, max(0, len(ids) - len(prefix) + 1)):
            if ids[i:i + len(prefix)] == prefix:
                start = i + len(prefix)
        if start is None:
            labels[:] = -100
        else:
            labels[:start] = -100
        labels[labels == self.tokenizer.pad_token_id] = -100
        return labels

    def __call__(self, batch):
        texts = []
        images = []
        task_types = []
        for example in batch:
            texts.append(self.processor.apply_chat_template(make_messages(example, include_answer=True), tokenize=False, add_generation_prompt=False))
            images.append([load_rgb(path) for path in example["image_paths"]])
            task_types.append(example["task_type"])
        encoded = self.processor(text=texts, images=images, padding=True, return_tensors="pt")
        encoded["labels"] = torch.stack([self._mask_prompt(row) for row in encoded["input_ids"]])
        encoded["task_type"] = task_types
        return encoded


In [ ]:
# 5) Load checkpoint-1200 trainable adapter and define memory-safe on-policy RL helpers
import torch.nn.functional as F
import bitsandbytes as bnb
from peft import PeftModel


_PHASE2_CHECKPOINT_PATTERN = re.compile(
    r"phase2-step-(\d+)-absolute-(\d+)$"
)


def find_latest_phase2_checkpoint():
    candidates = []
    for path in CHECKPOINT_ROOT.glob(
        "phase2-step-*-absolute-*"
    ):
        if not path.is_dir():
            continue
        match = _PHASE2_CHECKPOINT_PATTERN.search(
            path.name
        )
        if match is None:
            continue
        training_state_path = path / "training_state.pt"
        if not training_state_path.exists():
            continue
        candidates.append(
            (
                int(match.group(1)),
                int(match.group(2)),
                path,
            )
        )
    if not candidates:
        return None, 0, SOURCE_ABSOLUTE_RL_STEP
    candidates.sort(key=lambda item: item[0])
    phase_step, absolute_step, path = candidates[-1]
    return path, phase_step, absolute_step


if AUTO_RESUME_PHASE2:
    (
        LATEST_PHASE2_CHECKPOINT,
        RESUME_PHASE_STEP,
        RESUME_ABSOLUTE_STEP,
    ) = find_latest_phase2_checkpoint()
else:
    LATEST_PHASE2_CHECKPOINT = None
    RESUME_PHASE_STEP = 0
    RESUME_ABSOLUTE_STEP = SOURCE_ABSOLUTE_RL_STEP

LOAD_ADAPTER_DIR = (
    LATEST_PHASE2_CHECKPOINT
    if LATEST_PHASE2_CHECKPOINT is not None
    else SOURCE_RL_CHECKPOINT_DIR
)

print("Loading adapter from:", LOAD_ADAPTER_DIR)
print("Resume phase step:", RESUME_PHASE_STEP)
print("Resume absolute step:", RESUME_ABSOLUTE_STEP)

# Free stale objects before loading the 7B base model.
for _name in ["trainer", "model", "base_model", "processor", "optimizer"]:
    if _name in globals():
        del globals()[_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f"GPU memory before model load: {free / (1024 ** 3):.1f} GB free / {total / (1024 ** 3):.1f} GB total")

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
    local_files_only=MODEL_LOCAL_FILES_ONLY,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
processor.tokenizer.padding_side = "right"
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    local_files_only=MODEL_LOCAL_FILES_ONLY,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(
    base_model,
    use_gradient_checkpointing=True,
)

model = PeftModel.from_pretrained(
    base_model,
    LOAD_ADAPTER_DIR,
    is_trainable=True,
)
model.config.use_cache = False
model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)
model.enable_input_require_grads()

# Keep checkpointing active via model.train(), while disabling stochastic
# dropout so rollout scoring and policy-gradient rescoring use the same policy.
for module in model.modules():
    if isinstance(module, torch.nn.Dropout):
        module.p = 0.0

model.print_trainable_parameters()

optimizer = bnb.optim.PagedAdamW8bit(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
)
collator = PilotCCollator(processor)
rng = np.random.default_rng(SEED)

# The source step-150 weights start with a fresh optimizer because the sampler
# and reward objective changed. Only a checkpoint from this phase restores the
# phase-2 optimizer and RNG states.
if LATEST_PHASE2_CHECKPOINT is not None:
    training_state = torch.load(
        LATEST_PHASE2_CHECKPOINT / "training_state.pt",
        map_location="cpu",
        weights_only=False,
    )
    optimizer.load_state_dict(
        training_state["optimizer_state_dict"]
    )
    if training_state.get("torch_rng_state") is not None:
        torch.set_rng_state(
            training_state["torch_rng_state"]
        )
    if (
        torch.cuda.is_available()
        and training_state.get("cuda_rng_state_all") is not None
    ):
        torch.cuda.set_rng_state_all(
            training_state["cuda_rng_state_all"]
        )
    if training_state.get("numpy_rng_state") is not None:
        rng.bit_generator.state = training_state[
            "numpy_rng_state"
        ]
    print(
        "Restored phase-2 optimizer/RNG from:",
        LATEST_PHASE2_CHECKPOINT,
    )
else:
    print(
        "Fresh phase-2 optimizer at LR =",
        LEARNING_RATE,
    )


def model_device(active_model):
    return next(active_model.parameters()).device


def batch_to_device(inputs, active_model):
    return {
        key: value.to(model_device(active_model))
        if torch.is_tensor(value)
        else value
        for key, value in inputs.items()
    }


def row_to_base(row_index, row):
    return base_record(int(row_index), row)


def example_from_base_for_order(base):
    item = copy.deepcopy(base)
    item.update({
        "task_type": "order",
        "target": compact_order(base["order"]),
    })
    return item


def order_logprob_scores(
    active_model,
    example,
    candidate_orders,
    batch_size,
    grad_enabled,
):
    """
    Mean answer-token log probability for each candidate order.

    Rollout distribution:
        grad_enabled=False over all 24 candidates.

    Policy-gradient update:
        grad_enabled=True only for sampled candidates.
    """
    batch_size = int(batch_size)
    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "right"

    context = torch.enable_grad() if grad_enabled else torch.no_grad()

    try:
        prompt = processor.apply_chat_template(
            make_messages(example, include_answer=False),
            tokenize=False,
            add_generation_prompt=True,
        )
        images = [
            load_rgb(path)
            for path in example["image_paths"]
        ]
        prompt_inputs = processor(
            text=[prompt],
            images=[images],
            return_tensors="pt",
        )
        prompt_len = int(
            prompt_inputs["attention_mask"][0].sum().item()
        )

        scores = []
        orders = list(candidate_orders)

        with context:
            for start_index in range(
                0,
                len(orders),
                batch_size,
            ):
                batch_orders = orders[
                    start_index:start_index + batch_size
                ]
                texts = [
                    prompt + compact_order(order)
                    for order in batch_orders
                ]
                batch_images = [
                    images for _ in batch_orders
                ]
                inputs = processor(
                    text=texts,
                    images=batch_images,
                    padding=True,
                    return_tensors="pt",
                )
                inputs = batch_to_device(
                    inputs,
                    active_model,
                )
                outputs = active_model(**inputs)

                for row_index, _order in enumerate(
                    batch_orders
                ):
                    input_ids = inputs["input_ids"][
                        row_index
                    ]
                    attention_len = int(
                        inputs["attention_mask"][
                            row_index
                        ].sum().item()
                    )
                    target_len = (
                        attention_len - prompt_len
                    )
                    target_ids = input_ids[
                        prompt_len:
                        prompt_len + target_len
                    ]
                    logits = outputs.logits[
                        row_index,
                        prompt_len - 1:
                        prompt_len - 1 + target_len,
                    ]
                    token_logprob = -F.cross_entropy(
                        logits,
                        target_ids,
                        reduction="mean",
                    )
                    scores.append(token_logprob)

                # In no-grad rollout scoring, release the full
                # vocabulary logits before the next candidate batch.
                if not grad_enabled:
                    del outputs, inputs

        return torch.stack(scores)

    finally:
        processor.tokenizer.padding_side = (
            old_padding_side
        )


def reward_components(pred_order, gold_order):
    pred_order = [int(x) for x in pred_order]
    gold_order = [int(x) for x in gold_order]

    exact = float(pred_order == gold_order)
    first = float(
        pred_order[0] == gold_order[0]
    )
    last = float(
        pred_order[-1] == gold_order[-1]
    )
    both_endpoints = float(
        first == 1.0 and last == 1.0
    )

    gold_edges = {
        (gold_order[i], gold_order[i + 1])
        for i in range(3)
    }
    pred_edges = {
        (pred_order[i], pred_order[i + 1])
        for i in range(3)
    }
    adjacency = (
        len(gold_edges & pred_edges) / 3.0
    )

    pred_rank = {
        frame: idx
        for idx, frame in enumerate(pred_order)
    }
    gold_rank = {
        frame: idx
        for idx, frame in enumerate(gold_order)
    }
    pair_accuracy = float(np.mean([
        (
            pred_rank[a] < pred_rank[b]
        ) == (
            gold_rank[a] < gold_rank[b]
        )
        for a, b in itertools.combinations(
            [1, 2, 3, 4],
            2,
        )
    ]))

    total = (
        REWARD_WEIGHTS["exact"] * exact
        + REWARD_WEIGHTS["first"] * first
        + REWARD_WEIGHTS["last"] * last
        + REWARD_WEIGHTS["both_endpoints"] * both_endpoints
        + REWARD_WEIGHTS["adjacency"] * adjacency
        + REWARD_WEIGHTS["pair"] * pair_accuracy
    )

    return {
        "reward": float(total),
        "exact": exact,
        "first": first,
        "last": last,
        "both_endpoints": both_endpoints,
        "adjacency": float(adjacency),
        "pair": pair_accuracy,
    }


def select_gold_hard_negative_indices(
    policy_probs,
    gold_index,
    hard_negative_count=HARD_NEGATIVES_PER_SAMPLE,
    random_negative_count=RANDOM_NEGATIVES_PER_SAMPLE,
):
    """
    Candidate set for one training example:
      1 gold + top-k current-policy non-gold + sampled remaining non-gold.

    The hard negatives are recomputed from the current model every step.
    """
    probs = (
        policy_probs.detach()
        .float()
        .cpu()
        .numpy()
    )
    probs = probs / probs.sum()

    ranked_indices = [
        int(index)
        for index in np.argsort(-probs)
    ]
    non_gold_ranked = [
        index
        for index in ranked_indices
        if index != int(gold_index)
    ]

    hard_indices = non_gold_ranked[
        :int(hard_negative_count)
    ]
    chosen = [int(gold_index), *hard_indices]

    remaining = [
        index
        for index in non_gold_ranked
        if index not in set(hard_indices)
    ]
    random_indices = []

    random_count = min(
        int(random_negative_count),
        len(remaining),
    )
    if random_count > 0:
        remaining_probs = np.array(
            [probs[index] for index in remaining],
            dtype=np.float64,
        )
        if float(remaining_probs.sum()) <= 0:
            remaining_probs = np.full(
                len(remaining),
                1.0 / len(remaining),
            )
        else:
            remaining_probs = (
                remaining_probs
                / remaining_probs.sum()
            )
        sampled = rng.choice(
            len(remaining),
            size=random_count,
            replace=False,
            p=remaining_probs,
        )
        random_indices = [
            int(remaining[int(position)])
            for position in np.atleast_1d(sampled)
        ]
        chosen.extend(random_indices)

    if len(chosen) != len(set(chosen)):
        raise RuntimeError(
            f"Duplicate rollout candidates: {chosen}"
        )

    return chosen, {
        "gold_index": int(gold_index),
        "gold_rank": int(
            ranked_indices.index(int(gold_index)) + 1
        ),
        "hard_negative_indices": hard_indices,
        "random_negative_indices": random_indices,
    }


def prepare_rl_rollout(active_model, base):
    """
    1. Score all 24 actions without gradients.
    2. Sample current-policy actions.
    3. Compute deterministic gold rewards.
    4. Return zero-mean group advantages.

    The model is in eval mode during both rollout and RL scoring,
    so LoRA dropout does not make the rollout policy differ from
    the policy used for the gradient update.
    """
    active_model.eval()
    example = example_from_base_for_order(base)

    policy_scores = order_logprob_scores(
        active_model,
        example,
        PERMUTATIONS,
        batch_size=RL_CANDIDATE_BATCH_SIZE,
        grad_enabled=False,
    )
    policy_probs = torch.softmax(
        policy_scores.detach().float()
        / TEMPERATURE,
        dim=0,
    )
    entropy = float(
        -(
            policy_probs
            * torch.log(
                policy_probs.clamp_min(1e-12)
            )
        ).sum().item()
    )

    gold_tuple = tuple(
        int(value) for value in base["order"]
    )
    gold_index = PERMUTATIONS.index(
        gold_tuple
    )
    (
        sampled_indices,
        selection_meta,
    ) = select_gold_hard_negative_indices(
        policy_probs,
        gold_index,
    )
    if len(sampled_indices) != ROLLOUTS_PER_SAMPLE:
        raise RuntimeError(
            "Expected "
            f"{ROLLOUTS_PER_SAMPLE} rollout candidates, "
            f"got {len(sampled_indices)}"
        )
    sampled_orders = [
        list(PERMUTATIONS[index])
        for index in sampled_indices
    ]
    components = [
        reward_components(
            order,
            base["order"],
        )
        for order in sampled_orders
    ]

    rewards = torch.tensor(
        [component["reward"] for component in components],
        dtype=torch.float32,
        device=model_device(active_model),
    )
    reward_mean = rewards.mean()
    reward_std = rewards.std(unbiased=False)

    if float(reward_std.item()) < 1e-8:
        advantages = torch.zeros_like(rewards)
    else:
        advantages = (
            rewards - reward_mean
        ) / (
            reward_std + 1e-6
        )
        advantages = advantages.clamp(
            -2.0,
            2.0,
        )

        # Re-center after clipping. With sum(A)=0, the
        # categorical log-normalizer term cancels:
        # sum A_i log pi_i = sum A_i * score_i / T.
        advantages = (
            advantages - advantages.mean()
        )

    logs = {
        "mean_total_reward": float(
            rewards.mean().item()
        ),
        "mean_exact_reward": float(np.mean([
            component["exact"]
            for component in components
        ])),
        "mean_first_reward": float(np.mean([
            component["first"]
            for component in components
        ])),
        "mean_last_reward": float(np.mean([
            component["last"]
            for component in components
        ])),
        "mean_both_endpoints_reward": float(np.mean([
            component["both_endpoints"]
            for component in components
        ])),
        "mean_adjacency_reward": float(np.mean([
            component["adjacency"]
            for component in components
        ])),
        "mean_pair_reward": float(np.mean([
            component["pair"]
            for component in components
        ])),
        "gold_rank_before_update": float(
            selection_meta["gold_rank"]
        ),
        "gold_probability_before_update": float(
            policy_probs[
                selection_meta["gold_index"]
            ].item()
        ),
        "gold_in_rollout": 1.0,
        "hard_negative_orders": "|".join(
            compact_order(PERMUTATIONS[index])
            for index in selection_meta[
                "hard_negative_indices"
            ]
        ),
        "random_negative_orders": "|".join(
            compact_order(PERMUTATIONS[index])
            for index in selection_meta[
                "random_negative_indices"
            ]
        ),
        "rollout_unique_count": float(
            len({
                tuple(order)
                for order in sampled_orders
            })
        ),
        "policy_entropy": entropy,
        "reward_std": float(
            reward_std.item()
        ),
        "advantage_std": float(
            advantages.std(
                unbiased=False
            ).item()
        ),
        "top1_order": compact_order(
            PERMUTATIONS[
                int(
                    torch.argmax(
                        policy_probs
                    ).item()
                )
            ]
        ),
    }

    return {
        "example": example,
        "sampled_orders": sampled_orders,
        "advantages": advantages.detach(),
        "logs": logs,
    }


def backward_rl_rollout(active_model, rollout):
    """
    Backpropagate one sampled candidate at a time.

    The model must be in train mode here so Transformers actually uses
    gradient checkpointing. Dropout has already been set to p=0.
    """
    active_model.train()
    sampled_orders = rollout["sampled_orders"]
    advantages = rollout["advantages"]
    count = max(1, len(sampled_orders))

    detached_loss_total = 0.0

    for order, advantage in zip(
        sampled_orders,
        advantages,
    ):
        if abs(float(advantage.item())) < 1e-12:
            continue

        candidate_score = order_logprob_scores(
            active_model,
            rollout["example"],
            [order],
            batch_size=RL_GRAD_CANDIDATE_BATCH_SIZE,
            grad_enabled=True,
        )[0]

        candidate_loss = -(
            advantage
            * candidate_score
            / TEMPERATURE
            / count
        )
        candidate_loss.backward()
        detached_loss_total += float(
            candidate_loss.detach().cpu().item()
        )

        del candidate_score, candidate_loss

    return detached_loss_total


def compute_sft_replay_loss(
    active_model,
    records,
    batch_size=SFT_REPLAY_BATCH_SIZE,
):
    if not records:
        return (
            torch.tensor(
                0.0,
                device=model_device(active_model),
            ),
            {
                "train_sft_replay_loss": 0.0,
            },
        )

    batch = [
        records[int(i)]
        for i in rng.integers(
            0,
            len(records),
            size=int(batch_size),
        )
    ]
    inputs = collator(batch)
    task_types = inputs.pop("task_type")
    inputs = batch_to_device(
        inputs,
        active_model,
    )
    outputs = active_model(**inputs)

    logits = (
        outputs.logits[..., :-1, :]
        .contiguous()
    )
    labels = (
        inputs["labels"][..., 1:]
        .contiguous()
    )

    token_losses = F.cross_entropy(
        logits.view(
            -1,
            logits.size(-1),
        ),
        labels.view(-1),
        reduction="none",
        ignore_index=-100,
    ).view(labels.shape)

    mask = labels.ne(-100)
    sample_losses = (
        token_losses * mask
    ).sum(dim=1) / (
        mask.sum(dim=1).clamp_min(1)
    )

    weights = torch.tensor(
        [
            TASK_LOSS_WEIGHTS.get(
                task,
                1.0,
            )
            for task in task_types
        ],
        dtype=sample_losses.dtype,
        device=sample_losses.device,
    )
    loss = (
        sample_losses * weights
    ).mean()

    return (
        loss,
        {
            "train_sft_replay_loss": float(
                loss.detach().cpu().item()
            ),
        },
    )


def build_hard_sample_indices():
    """
    Hard examples must come from TRAIN-only predictions.

    Do not use BASE_OUTPUT_DIR/full100 caches here: those are
    validation predictions, so they do not overlap with training
    IDs and would either silently fall back or leak validation
    information into training selection.
    """
    random_indices = list(
        range(len(training_df))
    )

    if TRAIN_HARD_CACHE_PATH is None:
        print(
            "TRAIN_HARD_CACHE_PATH is not set. "
            "Using random train sampling only."
        )
        return [], random_indices

    cache_path = Path(TRAIN_HARD_CACHE_PATH)
    if not cache_path.exists():
        raise FileNotFoundError(cache_path)

    with open(
        cache_path,
        "r",
        encoding="utf-8",
    ) as handle:
        records = json.load(handle)

    train_id_set = set(
        training_df["Id"].astype(str)
    )
    hard_ids = set()

    for record in records:
        sample_id = str(
            record.get("sample_id")
        )
        if sample_id not in train_id_set:
            raise ValueError(
                "Hard cache contains a non-train ID: "
                f"{sample_id}"
            )
        if (
            record.get("direct_order")
            != record.get("gold_order")
            or record.get("final_order")
            != record.get("gold_order")
        ):
            hard_ids.add(sample_id)

    hard_indices = [
        int(index)
        for index, row in training_df.iterrows()
        if str(row["Id"]) in hard_ids
    ]

    print(
        "hard/random pools:",
        len(hard_indices),
        len(random_indices),
    )
    return hard_indices, random_indices


hard_indices, random_indices = (
    build_hard_sample_indices()
)


def choose_training_row_index():
    use_hard = (
        hard_indices
        and rng.random() < HARD_SAMPLE_RATIO
    )
    pool = (
        hard_indices
        if use_hard
        else random_indices
    )
    return int(rng.choice(pool))


def save_rl_checkpoint(phase_step):
    phase_step = int(phase_step)
    absolute_step = (
        SOURCE_ABSOLUTE_RL_STEP
        + phase_step
    )
    save_dir = (
        CHECKPOINT_ROOT
        / (
            f"phase2-step-{phase_step:03d}"
            f"-absolute-{absolute_step}"
        )
    )
    save_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    model.save_pretrained(save_dir)
    processor.save_pretrained(save_dir)

    torch.save(
        {
            "phase_step": phase_step,
            "absolute_step": absolute_step,
            "source_checkpoint": str(
                SOURCE_RL_CHECKPOINT_DIR
            ),
            "optimizer_state_dict":
                optimizer.state_dict(),
            "torch_rng_state":
                torch.get_rng_state(),
            "cuda_rng_state_all":
                torch.cuda.get_rng_state_all()
                if torch.cuda.is_available()
                else None,
            "numpy_rng_state":
                rng.bit_generator.state,
            "reward_weights":
                REWARD_WEIGHTS,
        },
        save_dir / "training_state.pt",
    )

    print("saved:", save_dir)
    return save_dir


TRAIN_LOG_PATH = OUTPUT_DIR / "rl_phase2_train_log.csv"
if TRAIN_LOG_PATH.exists():
    train_log_rows = pd.read_csv(
        TRAIN_LOG_PATH
    ).to_dict("records")
    print(
        "Loaded existing phase-2 train log:",
        len(train_log_rows),
        "rows",
    )
else:
    train_log_rows = []


def flush_train_logs():
    if not train_log_rows:
        return
    pd.DataFrame(train_log_rows).to_csv(
        TRAIN_LOG_PATH,
        index=False,
    )


In [ ]:
# 6) Fixed matched-20 direct evaluation with partial confidence metrics
import hashlib


def _normalized_distribution(values):
    values = np.asarray(values, dtype=np.float64)
    values = np.clip(values, 0.0, None)
    total = float(values.sum())
    if total <= 0:
        return np.full(
            len(values),
            1.0 / len(values),
            dtype=np.float64,
        )
    return values / total


def confidence_stats(values):
    probabilities = _normalized_distribution(values)
    order = np.argsort(-probabilities)
    top1 = float(probabilities[int(order[0])])
    top2 = float(probabilities[int(order[1])])
    entropy = float(
        -np.sum(
            probabilities
            * np.log(
                np.clip(probabilities, 1e-12, None)
            )
        )
    )
    max_entropy = math.log(len(probabilities))
    return {
        "probabilities": probabilities,
        "top1_probability": top1,
        "top2_probability": top2,
        "margin": top1 - top2,
        "entropy": entropy,
        "normalized_entropy": (
            entropy / max_entropy
            if max_entropy > 0
            else 0.0
        ),
    }


def pair_accuracy_for_order(pred_order, gold_order):
    pred_rank = {
        frame: index
        for index, frame in enumerate(pred_order)
    }
    gold_rank = {
        frame: index
        for index, frame in enumerate(gold_order)
    }
    return float(np.mean([
        (
            pred_rank[a] < pred_rank[b]
        ) == (
            gold_rank[a] < gold_rank[b]
        )
        for a, b in itertools.combinations(
            [1, 2, 3, 4],
            2,
        )
    ]))


def evaluate_one_quick_partial(
    active_model,
    row_index,
    row,
):
    base = row_to_base(row_index, row)
    example = example_from_base_for_order(base)

    scores = order_logprob_scores(
        active_model,
        example,
        PERMUTATIONS,
        batch_size=QUICK_EVAL_CANDIDATE_BATCH_SIZE,
        grad_enabled=False,
    ).detach().float().cpu()

    # Match full100 direct evaluation: no temperature scaling here.
    probabilities = torch.softmax(
        scores,
        dim=0,
    ).numpy()

    pred_index = int(np.argmax(probabilities))
    pred_order = list(PERMUTATIONS[pred_index])
    gold_order = [int(value) for value in base["order"]]
    gold_index = PERMUTATIONS.index(tuple(gold_order))

    full_stats = confidence_stats(probabilities)

    frame_labels = [1, 2, 3, 4]
    first_probs = np.array([
        sum(
            probability
            for order, probability in zip(
                PERMUTATIONS,
                probabilities,
            )
            if int(order[0]) == frame
        )
        for frame in frame_labels
    ])
    last_probs = np.array([
        sum(
            probability
            for order, probability in zip(
                PERMUTATIONS,
                probabilities,
            )
            if int(order[-1]) == frame
        )
        for frame in frame_labels
    ])
    first_stats = confidence_stats(first_probs)
    last_stats = confidence_stats(last_probs)

    selected_first = int(pred_order[0])
    selected_last = int(pred_order[-1])

    last_given_first_labels = [
        frame for frame in frame_labels
        if frame != selected_first
    ]
    last_given_first_probs = np.array([
        sum(
            probability
            for order, probability in zip(
                PERMUTATIONS,
                probabilities,
            )
            if (
                int(order[0]) == selected_first
                and int(order[-1]) == frame
            )
        )
        for frame in last_given_first_labels
    ])
    last_given_first_stats = confidence_stats(
        last_given_first_probs
    )

    first_given_last_labels = [
        frame for frame in frame_labels
        if frame != selected_last
    ]
    first_given_last_probs = np.array([
        sum(
            probability
            for order, probability in zip(
                PERMUTATIONS,
                probabilities,
            )
            if (
                int(order[-1]) == selected_last
                and int(order[0]) == frame
            )
        )
        for frame in first_given_last_labels
    ])
    first_given_last_stats = confidence_stats(
        first_given_last_probs
    )

    endpoint_pairs = [
        (first, last)
        for first in frame_labels
        for last in frame_labels
        if first != last
    ]
    endpoint_pair_probs = np.array([
        sum(
            probability
            for order, probability in zip(
                PERMUTATIONS,
                probabilities,
            )
            if (
                int(order[0]) == first
                and int(order[-1]) == last
            )
        )
        for first, last in endpoint_pairs
    ])
    endpoint_stats = confidence_stats(
        endpoint_pair_probs
    )

    middle_indices = [
        index
        for index, order in enumerate(PERMUTATIONS)
        if (
            int(order[0]) == selected_first
            and int(order[-1]) == selected_last
        )
    ]
    middle_probs = np.array([
        probabilities[index]
        for index in middle_indices
    ])
    middle_stats = confidence_stats(middle_probs)

    exact = int(pred_order == gold_order)
    first_correct = int(
        pred_order[0] == gold_order[0]
    )
    last_correct = int(
        pred_order[-1] == gold_order[-1]
    )
    both_correct = int(
        first_correct == 1
        and last_correct == 1
    )

    sorted_indices = np.argsort(-probabilities)
    gold_rank = int(
        np.where(sorted_indices == gold_index)[0][0]
        + 1
    )

    return {
        "sample_id": str(row["Id"]),
        "gold_order": compact_order(gold_order),
        "pred_order": compact_order(pred_order),
        "exact": exact,
        "first_correct": first_correct,
        "last_correct": last_correct,
        "both_endpoints_correct": both_correct,
        "relative_pair_accuracy": pair_accuracy_for_order(
            pred_order,
            gold_order,
        ),
        "gold_probability": float(
            probabilities[gold_index]
        ),
        "gold_rank": gold_rank,

        "full_top1_probability": full_stats[
            "top1_probability"
        ],
        "full_top1_margin": full_stats["margin"],
        "full_policy_entropy": full_stats["entropy"],
        "full_normalized_entropy": full_stats[
            "normalized_entropy"
        ],

        "first_marginal_top1_probability": first_stats[
            "top1_probability"
        ],
        "first_marginal_margin": first_stats["margin"],
        "first_marginal_normalized_entropy": first_stats[
            "normalized_entropy"
        ],

        "last_marginal_top1_probability": last_stats[
            "top1_probability"
        ],
        "last_marginal_margin": last_stats["margin"],
        "last_marginal_normalized_entropy": last_stats[
            "normalized_entropy"
        ],

        "last_given_first_top1_probability": (
            last_given_first_stats["top1_probability"]
        ),
        "last_given_first_margin": last_given_first_stats[
            "margin"
        ],
        "last_given_first_normalized_entropy": (
            last_given_first_stats["normalized_entropy"]
        ),

        "first_given_last_top1_probability": (
            first_given_last_stats["top1_probability"]
        ),
        "first_given_last_margin": first_given_last_stats[
            "margin"
        ],
        "first_given_last_normalized_entropy": (
            first_given_last_stats["normalized_entropy"]
        ),

        "endpoint_pair_top1_probability": endpoint_stats[
            "top1_probability"
        ],
        "endpoint_pair_margin": endpoint_stats["margin"],
        "endpoint_pair_normalized_entropy": endpoint_stats[
            "normalized_entropy"
        ],

        "middle_top1_probability": middle_stats[
            "top1_probability"
        ],
        "middle_margin": middle_stats["margin"],
        "middle_normalized_entropy": middle_stats[
            "normalized_entropy"
        ],

        # Current global fallback diagnostic only; not applied to predictions.
        "global_low_confidence": int(
            full_stats["entropy"] > 3.0
            and full_stats["margin"] < 0.01
        ),
    }


def _mean_by_correctness(dataframe, metric, target):
    correct = dataframe.loc[
        dataframe[target] == 1,
        metric,
    ]
    wrong = dataframe.loc[
        dataframe[target] == 0,
        metric,
    ]
    return (
        float(correct.mean())
        if len(correct) else np.nan,
        float(wrong.mean())
        if len(wrong) else np.nan,
    )


def run_quick20_partial_eval(
    active_model,
    phase_step,
    force=FORCE_QUICK_REEVAL,
):
    phase_step = int(phase_step)
    absolute_step = (
        SOURCE_ABSOLUTE_RL_STEP
        + phase_step
    )
    label = (
        f"phase2_step_{phase_step:03d}"
        f"_absolute_{absolute_step}"
    )
    detail_path = QUICK_EVAL_DIR / f"{label}_detail.csv"
    summary_path = QUICK_EVAL_DIR / f"{label}_summary.csv"

    if summary_path.exists() and not force:
        print(
            "Quick evaluation already exists; skipping:",
            summary_path,
        )
        return pd.read_csv(summary_path)

    previous_mode = active_model.training
    active_model.eval()

    target_ids = set(
        quick_eval_df["Id"].astype(str).tolist()
    )
    if detail_path.exists() and not force:
        cached_detail_df = pd.read_csv(detail_path)
        cached_detail_df["sample_id"] = (
            cached_detail_df["sample_id"].astype(str)
        )
        cached_ids = set(
            cached_detail_df["sample_id"].tolist()
        )
        if not cached_ids.issubset(target_ids):
            raise RuntimeError(
                "Quick-eval cache contains IDs outside the fixed quick20 set: "
                f"{detail_path}"
            )
        rows = cached_detail_df.to_dict("records")
        completed_ids = cached_ids
        print(
            "Resuming quick evaluation:",
            len(rows),
            "/",
            len(quick_eval_df),
        )
    else:
        rows = []
        completed_ids = set()

    try:
        progress = tqdm(
            quick_eval_df.iterrows(),
            total=len(quick_eval_df),
            desc=f"quick20 {label}",
        )
        for row_index, row in progress:
            sample_id = str(row["Id"])
            if sample_id in completed_ids:
                continue
            result = evaluate_one_quick_partial(
                active_model,
                int(row_index),
                row,
            )
            rows.append(result)
            completed_ids.add(sample_id)
            pd.DataFrame(rows).to_csv(
                detail_path,
                index=False,
            )
            progress.set_postfix({
                "exact": f"{np.mean([x['exact'] for x in rows]):.3f}",
                "done": len(rows),
            })
    finally:
        if previous_mode:
            active_model.train()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    detail_df = pd.DataFrame(rows)

    first_margin_correct, first_margin_wrong = (
        _mean_by_correctness(
            detail_df,
            "first_marginal_margin",
            "first_correct",
        )
    )
    last_margin_correct, last_margin_wrong = (
        _mean_by_correctness(
            detail_df,
            "last_marginal_margin",
            "last_correct",
        )
    )
    endpoint_margin_correct, endpoint_margin_wrong = (
        _mean_by_correctness(
            detail_df,
            "endpoint_pair_margin",
            "both_endpoints_correct",
        )
    )
    full_margin_correct, full_margin_wrong = (
        _mean_by_correctness(
            detail_df,
            "full_top1_margin",
            "exact",
        )
    )

    low_conf_df = detail_df[
        detail_df["global_low_confidence"] == 1
    ]

    summary = pd.DataFrame([{
        "phase_step": phase_step,
        "absolute_step": absolute_step,
        "samples": len(detail_df),
        "quick_eval_seed": MATCHED_EVAL_SEED,
        "quick_eval_sample_hash": quick_eval_sample_hash,
        "direct_exact": float(detail_df["exact"].mean()),
        "first_accuracy": float(
            detail_df["first_correct"].mean()
        ),
        "last_accuracy": float(
            detail_df["last_correct"].mean()
        ),
        "both_endpoints": float(
            detail_df["both_endpoints_correct"].mean()
        ),
        "relative_pair_accuracy": float(
            detail_df["relative_pair_accuracy"].mean()
        ),
        "mean_gold_probability": float(
            detail_df["gold_probability"].mean()
        ),
        "mean_gold_rank": float(
            detail_df["gold_rank"].mean()
        ),
        "mean_full_top1_probability": float(
            detail_df["full_top1_probability"].mean()
        ),
        "mean_full_top1_margin": float(
            detail_df["full_top1_margin"].mean()
        ),
        "mean_full_policy_entropy": float(
            detail_df["full_policy_entropy"].mean()
        ),
        "mean_first_marginal_margin": float(
            detail_df["first_marginal_margin"].mean()
        ),
        "mean_first_marginal_normalized_entropy": float(
            detail_df[
                "first_marginal_normalized_entropy"
            ].mean()
        ),
        "mean_last_marginal_margin": float(
            detail_df["last_marginal_margin"].mean()
        ),
        "mean_last_marginal_normalized_entropy": float(
            detail_df[
                "last_marginal_normalized_entropy"
            ].mean()
        ),
        "mean_last_given_first_margin": float(
            detail_df["last_given_first_margin"].mean()
        ),
        "mean_last_given_first_normalized_entropy": float(
            detail_df[
                "last_given_first_normalized_entropy"
            ].mean()
        ),
        "mean_first_given_last_margin": float(
            detail_df["first_given_last_margin"].mean()
        ),
        "mean_first_given_last_normalized_entropy": float(
            detail_df[
                "first_given_last_normalized_entropy"
            ].mean()
        ),
        "mean_endpoint_pair_margin": float(
            detail_df["endpoint_pair_margin"].mean()
        ),
        "mean_endpoint_pair_normalized_entropy": float(
            detail_df[
                "endpoint_pair_normalized_entropy"
            ].mean()
        ),
        "mean_middle_margin": float(
            detail_df["middle_margin"].mean()
        ),
        "mean_middle_normalized_entropy": float(
            detail_df[
                "middle_normalized_entropy"
            ].mean()
        ),
        "full_margin_correct_mean": full_margin_correct,
        "full_margin_wrong_mean": full_margin_wrong,
        "first_margin_correct_mean": first_margin_correct,
        "first_margin_wrong_mean": first_margin_wrong,
        "last_margin_correct_mean": last_margin_correct,
        "last_margin_wrong_mean": last_margin_wrong,
        "endpoint_margin_correct_mean": endpoint_margin_correct,
        "endpoint_margin_wrong_mean": endpoint_margin_wrong,
        "global_low_confidence_count": int(
            detail_df["global_low_confidence"].sum()
        ),
        "global_low_confidence_exact": (
            float(low_conf_df["exact"].mean())
            if len(low_conf_df)
            else np.nan
        ),
    }])

    summary.to_csv(summary_path, index=False)

    all_summary_paths = sorted(
        QUICK_EVAL_DIR.glob(
            "phase2_step_*_summary.csv"
        )
    )
    all_summaries = pd.concat(
        [pd.read_csv(path) for path in all_summary_paths],
        ignore_index=True,
    ).sort_values("phase_step")
    all_summaries.to_csv(
        QUICK_EVAL_DIR / "quick20_all_checkpoints_summary.csv",
        index=False,
    )

    print("\nQuick20 result:")
    display(summary)
    print("detail :", detail_path)
    print("summary:", summary_path)
    return summary


In [ ]:
# 7) Run phase-2 training, checkpoint every 25 steps, and quick-evaluate

# Baseline quick20 at the unmodified source step-150 adapter.
if (
    RUN_INITIAL_QUICK_EVAL
    and RESUME_PHASE_STEP == 0
):
    run_quick20_partial_eval(
        model,
        phase_step=0,
    )

# On a resumed runtime, finish a missing quick evaluation for the last
# already-saved phase checkpoint before continuing training.
if RESUME_PHASE_STEP > 0:
    run_quick20_partial_eval(
        model,
        phase_step=RESUME_PHASE_STEP,
    )

if RESUME_PHASE_STEP >= RL_STEPS:
    print(
        "Phase-2 training is already complete:",
        RESUME_PHASE_STEP,
        "/",
        RL_STEPS,
    )
else:
    for phase_step in range(
        RESUME_PHASE_STEP + 1,
        RL_STEPS + 1,
    ):
        absolute_step = (
            SOURCE_ABSOLUTE_RL_STEP
            + phase_step
        )

        row_index = choose_training_row_index()
        row = training_df.iloc[row_index]
        base = row_to_base(row_index, row)

        optimizer.zero_grad(set_to_none=True)

        # ----------------------------------------------------
        # A. Gold-aware hard-negative rollout and RL gradient
        # ----------------------------------------------------
        model.eval()
        rollout = prepare_rl_rollout(
            model,
            base,
        )

        # Switches to train mode for gradient checkpointing.
        # All Dropout modules have p=0.
        rl_loss_value = backward_rl_rollout(
            model,
            rollout,
        )

        # ----------------------------------------------------
        # B. Multitask SFT replay gradient
        # ----------------------------------------------------
        model.train()
        sft_loss, sft_logs = (
            compute_sft_replay_loss(
                model,
                train_records,
                batch_size=SFT_REPLAY_BATCH_SIZE,
            )
        )
        weighted_sft_loss = (
            SFT_REPLAY_WEIGHT * sft_loss
        )

        if not torch.isfinite(weighted_sft_loss):
            raise RuntimeError(
                "Non-finite SFT replay loss "
                f"at phase step {phase_step}: "
                f"{weighted_sft_loss}"
            )

        weighted_sft_loss.backward()

        trainable_parameters = [
            parameter
            for parameter in model.parameters()
            if parameter.requires_grad
        ]
        grad_norm = torch.nn.utils.clip_grad_norm_(
            trainable_parameters,
            MAX_GRAD_NORM,
        )

        if not torch.isfinite(
            torch.as_tensor(grad_norm)
        ):
            raise RuntimeError(
                "Non-finite grad norm at phase step "
                f"{phase_step}: {grad_norm}"
            )

        optimizer.step()

        total_loss_value = (
            float(rl_loss_value)
            + SFT_REPLAY_WEIGHT
            * float(sft_loss.detach().cpu().item())
        )

        log_row = {
            "phase_step": phase_step,
            "absolute_step": absolute_step,
            "sample_id": str(row["Id"]),
            "train_total_loss": float(total_loss_value),
            "train_rl_loss": float(rl_loss_value),
            "learning_rate": LEARNING_RATE,
            "grad_norm": float(
                grad_norm.detach().cpu().item()
                if torch.is_tensor(grad_norm)
                else grad_norm
            ),
            **rollout["logs"],
            **sft_logs,
        }
        train_log_rows.append(log_row)

        if (
            phase_step % LOGGING_STEPS == 0
            or phase_step == RESUME_PHASE_STEP + 1
        ):
            print(log_row)
            flush_train_logs()

        if phase_step % SAVE_STEPS == 0:
            flush_train_logs()
            save_dir = save_rl_checkpoint(
                phase_step
            )

            if RUN_QUICK_EVAL_DURING_TRAIN:
                run_quick20_partial_eval(
                    model,
                    phase_step=phase_step,
                )

            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    flush_train_logs()

    final_dir = OUTPUT_DIR / "final_phase2_adapter"
    final_dir.mkdir(
        parents=True,
        exist_ok=True,
    )
    model.save_pretrained(final_dir)
    processor.save_pretrained(final_dir)
    print("final phase-2 adapter saved:", final_dir)

print("\nAll quick checkpoint summaries:")
all_quick_summary_path = (
    QUICK_EVAL_DIR
    / "quick20_all_checkpoints_summary.csv"
)
if all_quick_summary_path.exists():
    display(pd.read_csv(all_quick_summary_path))
else:
    print("No quick summary file yet.")


In [ ]:
# 8) Review training and quick-evaluation trends without rerunning the model

if TRAIN_LOG_PATH.exists():
    phase2_train_log_df = pd.read_csv(TRAIN_LOG_PATH)
    print("Phase-2 training log tail:")
    display(phase2_train_log_df.tail(20))
else:
    print("Training log not found:", TRAIN_LOG_PATH)

quick_summary_path = (
    QUICK_EVAL_DIR
    / "quick20_all_checkpoints_summary.csv"
)
if quick_summary_path.exists():
    quick_checkpoint_summary_df = pd.read_csv(
        quick_summary_path
    ).sort_values("phase_step")
    print("Quick20 checkpoint comparison:")
    display(quick_checkpoint_summary_df)
else:
    print("Quick summary not found:", quick_summary_path)


## Full evaluation decision

Use the fixed quick20 results only to select one or two promising phase checkpoints. Then run the same matched full100 direct evaluation on those selected checkpoints.

The quick20 set is a fixed subset of the matched seed-43 validation 100, so all phase checkpoints are compared on identical samples.